In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_csv("vendors.csv")

print(df.head())
print(df.info())

               Company           Work_Type  Rating   Category
0  FinEdge Enterprises  Civil Construction     4.4     Design
1   TechNova Solutions     Interior Design     3.1     Design
2  SkyHigh Enterprises   Digital Marketing     4.3  Education
3   VisionTech Pvt Ltd   Digital Marketing     3.0  Marketing
4    CloudLink Pvt Ltd      Cyber Security     4.9  Education
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Company    10000 non-null  object 
 1   Work_Type  10000 non-null  object 
 2   Rating     10000 non-null  float64
 3   Category   10000 non-null  object 
dtypes: float64(1), object(3)
memory usage: 312.6+ KB
None


In [3]:
print(df.isnull().sum())

Company      0
Work_Type    0
Rating       0
Category     0
dtype: int64


In [4]:
df = df.dropna()

In [5]:
df["combined_text"] = df["Work_Type"] + " " + df["Category"]

In [6]:
vectorizer = TfidfVectorizer()

In [7]:
text_vectors = vectorizer.fit_transform(df["combined_text"])

In [8]:
print(text_vectors.shape)

(10000, 24)


In [9]:
scaler = MinMaxScaler()

In [10]:
rating_scaled = scaler.fit_transform(df[["Rating"]])

In [11]:
final_features = np.hstack((text_vectors.toarray(), rating_scaled))

In [12]:
similarity_matrix = cosine_similarity(final_features)

In [13]:
def recommend_vendor(company_name, df, similarity_matrix, top_n=3):

    # Get index of company
    index = df[df["Company"] == company_name].index[0]

    # Get similarity scores
    similarity_scores = list(enumerate(similarity_matrix[index]))

    # Sort descending
    sorted_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

    # Exclude itself
    sorted_scores = sorted_scores[1:top_n+1]

    # Get vendor indices
    vendor_indices = [i[0] for i in sorted_scores]

    return df.iloc[vendor_indices]

In [16]:
print(df[df["Company"] == "XYZ Solutions"])

Empty DataFrame
Columns: [Company, Work_Type, Rating, Category, combined_text]
Index: []


In [17]:
print(df["Company"].head(20))
print(df["Company"].unique()[:20])

0         FinEdge Enterprises
1          TechNova Solutions
2         SkyHigh Enterprises
3          VisionTech Pvt Ltd
4           CloudLink Pvt Ltd
5       GreenLeaf Enterprises
6        BuildPro Enterprises
7          AdWise Enterprises
8      InteriorHive Solutions
9       BrightPath Industries
10      CoreBuild Enterprises
11     CyberMatrix Industries
12           VisionTech Group
13          CoreBuild Pvt Ltd
14       MarketGenius Pvt Ltd
15          FinEdge Solutions
16         BuildPro Solutions
17     MarketGenius Solutions
18     CyberMatrix Industries
19    MarketGenius Industries
Name: Company, dtype: object
['FinEdge Enterprises' 'TechNova Solutions' 'SkyHigh Enterprises'
 'VisionTech Pvt Ltd' 'CloudLink Pvt Ltd' 'GreenLeaf Enterprises'
 'BuildPro Enterprises' 'AdWise Enterprises' 'InteriorHive Solutions'
 'BrightPath Industries' 'CoreBuild Enterprises' 'CyberMatrix Industries'
 'VisionTech Group' 'CoreBuild Pvt Ltd' 'MarketGenius Pvt Ltd'
 'FinEdge Solutions' 'BuildPro S

In [18]:
recommend_vendor("FinEdge Solutions", df, similarity_matrix)

,Company,Work_Type,Rating,Category,combined_text
863,BuildPro Group,Civil Construction,4.3,Finance,Civil Construction Finance
3138,GreenLeaf Solutions,Civil Construction,4.3,Finance,Civil Construction Finance
3617,CloudLink Solutions,Civil Construction,4.3,Finance,Civil Construction Finance


In [21]:
def recommend_for_new_project(work, category, rating):

    input_text = work + " " + category

    # Convert text
    input_vector = vectorizer.transform([input_text])

    # Scale rating
    input_rating = scaler.transform([[rating]])

    # Combine
    input_final = np.hstack((input_vector.toarray(), input_rating))

    # Calculate similarity
    similarities = cosine_similarity(input_final, final_features)

    # Sort
    sorted_scores = sorted(list(enumerate(similarities[0])), key=lambda x: x[1], reverse=True)

    top_indices = [i[0] for i in sorted_scores[:3]]

    return df.iloc[top_indices]

In [22]:
recommend_for_new_project("AI ML", "IT", 4.6)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


,Company,Work_Type,Rating,Category,combined_text
2918,PrimeLogics Pvt Ltd,AI/ML Services,5.0,IT Services,AI/ML Services IT Services
4596,CyberMatrix Pvt Ltd,AI/ML Services,5.0,IT Services,AI/ML Services IT Services
9651,BrightPath Enterprises,AI/ML Services,5.0,IT Services,AI/ML Services IT Services


In [23]:
final_features = np.hstack((text_vectors.toarray()*0.8, rating_scaled*0.2))

In [24]:
import pickle

pickle.dump(vectorizer, open("vectorizer.pkl", "wb"))
pickle.dump(scaler, open("scaler.pkl", "wb"))
pickle.dump(final_features, open("features.pkl", "wb"))